# 🛡️ DawnGNN: Documentation-Augmented Graph Neural Networks for Windows Malware Detection

Welcome to the interactive demonstration of **DawnGNN**. This notebook guides you end-to-end through:
1. **Dataset Exploration**: Inspecting dynamic Windows API call sequences from the `MalBehavD-V1` dataset.
2. **Graph Modeling**: Transforming sequential execution traces into directed behavioral graphs.
3. **Semantic Augmentation**: Enhancing nodes with Microsoft Win32 documentation embeddings via BERT (768-d).
4. **Inference with GAT**: Running our pre-trained Graph Attention Network (`GATClassifier`) on benign vs malware samples.
5. **Benchmark Comparison**: Evaluating DawnGNN against state-of-the-art baselines (*MalDetConv*, *MalDy*, and ablation variants).

---

## 1. Setup & Environment Verification

In [ ]:
import sys
from pathlib import Path

# Add src/ to Python path
PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import json
import torch
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

from utils.graph_utils import build_graph_from_api_list, normalize_api
from utils.paths import get_project_root, get_model_path, get_data_path

print(f"[✓] PyTorch version: {torch.__version__}")
print(f"[✓] CUDA available:  {torch.cuda.is_available()}")
print(f"[✓] Project root:    {PROJECT_ROOT}")

## 2. Exploring the Raw Malware Dataset (MalBehavD-V1)

The raw dataset records sequential dynamic execution traces collected in a sandbox environment.

In [ ]:
csv_path = PROJECT_ROOT / "data" / "raw" / "MalBehavD-V1-dataset.csv"

if csv_path.exists():
    df_preview = pd.read_csv(csv_path, nrows=5)
    print(f"Dataset Shape (preview): {df_preview.shape}")
    print(f"Columns: {list(df_preview.columns[:10])} ...")
    display(df_preview.iloc[:, :6])
else:
    print(f"Raw CSV file not found at {csv_path}. Using synthetic sample trace for demonstration.")

## 3. Behavioral Graph Construction

We convert an ordered API sequence into a directed call transition graph $G = (V, E)$, where:
- **Nodes ($V$)**: Unique Windows API functions invoked during execution.
- **Edges ($E$)**: Directed transitions $(API_i \to API_{i+1})$ weighted by transition frequency.

In [ ]:
# Example malicious API call sequence (Ransomware-style behavior: discovery -> encryption -> deletion)
sample_api_sequence = [
    "NtAllocateVirtualMemory",
    "RegOpenKeyExW",
    "RegQueryValueExW",
    "FindFirstFileW",
    "FindNextFileW",
    "NtCreateFile",
    "NtReadFile",
    "CryptEncrypt",
    "NtWriteFile",
    "NtClose",
    "DeleteFileW",
    "NtTerminateProcess"
]

# Build NetworkX graph
G = build_graph_from_api_list(sample_api_sequence)
print(f"[✓] Constructed Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

# Visualize the graph
plt.figure(figsize=(10, 6))
pos = nx.spring_layout(G, seed=42)
nx.draw_networkx_nodes(G, pos, node_size=1600, node_color="#6366f1", alpha=0.9)
nx.draw_networkx_labels(G, pos, font_size=8, font_color="white", font_weight="bold")
nx.draw_networkx_edges(G, pos, edge_color="#64748b", arrows=True, arrowsize=15, width=1.5)
plt.title("Behavioral API Call Graph for Execution Trace", fontsize=12, pad=12)
plt.axis("off")
plt.tight_layout()
plt.show()

## 4. Semantic Augmentation with BERT

Instead of arbitrary one-hot IDs or static Word2Vec tokens, DawnGNN retrieves official Microsoft Win32 documentation and passes descriptions through **BERT** (`bert-base-uncased`) to create rich 768-dimensional initial node embeddings.

In [ ]:
docs_path = PROJECT_ROOT / "data" / "api_docs" / "api_docs.json"

if docs_path.exists():
    with open(docs_path, "r", encoding="utf-8") as f:
        api_docs = json.load(f)
    print(f"[✓] Total documented Windows APIs in knowledge base: {len(api_docs):,}")
    
    # Sample API doc preview
    sample_apis = ["RegOpenKeyExW", "CryptEncrypt", "NtTerminateProcess"]
    for api in sample_apis:
        desc = api_docs.get(api, "(No official doc found)")
        print(f"\n• API: {api}")
        print(f"  Documentation: {desc[:140]}...")
else:
    print("api_docs.json not found in data/api_docs/")

## 5. Loading Trained DawnGNN (GAT) Model

In [ ]:
from models.gat import GATClassifier

# Instantiate architecture
model = GATClassifier(
    in_channels=768,
    hidden_channels=128,
    out_channels=2,
    num_layers=3,
    num_heads=8,
    dropout=0.4,
    pooling_type="mean"
)

weights_path = PROJECT_ROOT / "models" / "gnn" / "best_model.pt"
if weights_path.exists():
    state = torch.load(weights_path, map_location="cpu")
    if "model_state_dict" in state:
        model.load_state_dict(state["model_state_dict"])
    else:
        model.load_state_dict(state)
    model.eval()
    print(f"[✓] Successfully loaded trained weights from {weights_path}")
else:
    print(f"Weights file not found at {weights_path}. Model instantiated with random weights for demo.")

total_params = sum(p.numel() for p in model.parameters())
print(f"Total Trainable Parameters: {total_params:,}")

## 6. End-to-End Inference Demo

We use the high-level `predict_sequence` pipeline from `src/serving/infer.py` to classify arbitrary raw API traces.

In [ ]:
from serving.infer import predict_sequence

# Sample traces
benign_trace = ["GetSystemTime", "GetVersionExW", "GetCommandLineW", "ExitProcess"]
malware_trace = sample_api_sequence

print("=== BENIGN SAMPLE INFERENCE ===")
res_benign = predict_sequence(benign_trace, explain=True)
print(f"Prediction:  {res_benign['prediction']} (Label: {res_benign['predicted_label']})")
print(f"Confidence:  {res_benign['confidence']:.2%}")
print(f"Probabilities: Benign={res_benign['probabilities']['benign']:.4f}, Malware={res_benign['probabilities']['malware']:.4f}")

print("\n=== MALWARE SAMPLE INFERENCE ===")
res_malware = predict_sequence(malware_trace, explain=True)
print(f"Prediction:  {res_malware['prediction']} (Label: {res_malware['predicted_label']})")
print(f"Confidence:  {res_malware['confidence']:.2%}")
print(f"Probabilities: Benign={res_malware['probabilities']['benign']:.4f}, Malware={res_malware['probabilities']['malware']:.4f}")

## 7. Experimental Benchmark Results

We compare **DawnGNN (BERT)** against 4 baseline methods on the test split ($N = 255$ samples):

In [ ]:
csv_results = PROJECT_ROOT / "results" / "evaluation_summary.csv"
if csv_results.exists():
    df_results = pd.read_csv(csv_results)
    display(df_results)
    
    # Bar chart comparison
    plt.figure(figsize=(10, 5))
    methods = df_results["Method"].tolist()
    acc = df_results["Accuracy"].tolist()
    f1 = df_results["F1-Score (Macro)"].tolist()
    
    x = np.arange(len(methods))
    width = 0.35
    
    plt.bar(x - width/2, acc, width, label="Accuracy", color="#4f46e5")
    plt.bar(x + width/2, f1, width, label="F1-Score", color="#06b6d4")
    
    plt.ylabel("Score")
    plt.title("DawnGNN vs Baselines Performance Comparison")
    plt.xticks(x, methods, rotation=15, ha="right")
    plt.ylim([0.4, 1.0])
    plt.legend()
    plt.grid(axis="y", linestyle="--", alpha=0.4)
    plt.tight_layout()
    plt.show()
else:
    print("Results CSV not found.")